# AI Detection Evaluator Comparison

This notebook evaluates the same input image folder with:
- SDXL detector (`AIDetectionImageEvaluator`, `Organika/sdxl-detector`)
- SSP (`SSPAIDetectionImageEvaluator`)
- DIRE official (`DIREAIDetectionImageEvaluator`, `adm-ddim-official`)
- DIRE SDXL adaptation (`DIREAIDetectionImageEvaluator`, `sdxl-turbo-experimental`)

Note: DIRE initialization can be slow on first run because it may download/load large checkpoints and pipelines.


In [ ]:
from pathlib import Path
import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import importlib

import evolutionary_model_helpers.auto_device as ad
import evolutionary_imaging.evaluators as evaluators_module
importlib.reload(evaluators_module)
from evolutionary_imaging.image_base import ImageSolutionData
from evolutionary_imaging.evaluators import (
    AIDetectionImageEvaluator,
    SSPAIDetectionImageEvaluator,
    DIREAIDetectionImageEvaluator,
)


In [ ]:
# Configuration
IMAGE_FOLDER = Path('../pre-testing/testimages')
DEVICE = ad.auto_device()
DIRE_DEVICE = DEVICE

# Optional local override for DIRE classifier checkpoint
DIRE_CLASSIFIER_CHECKPOINT_PATH = None
DIRE_DOWNLOAD_IF_MISSING = True

# Prefer a known local checkpoint path when present to avoid re-downloads
dire_ckpt_candidates = [
    Path('./pre-testing/models/dire/classifier/lsun_adm.pth'),
    Path('../pre-testing/models/dire/classifier/lsun_adm.pth'),
]
if DIRE_CLASSIFIER_CHECKPOINT_PATH is None:
    for candidate in dire_ckpt_candidates:
        if candidate.exists():
            DIRE_CLASSIFIER_CHECKPOINT_PATH = str(candidate)
            break

# Toggle evaluators
ENABLE_SDXL_DETECTOR = True
ENABLE_SSP = True
ENABLE_DIRE_OFFICIAL = True
ENABLE_DIRE_SDXL_EXPERIMENTAL = True

print(f'Working directory: {Path.cwd()}')
print(f'Using device: {DEVICE}')
print(f'Using DIRE device request: {DIRE_DEVICE}')
print(f'Image folder: {IMAGE_FOLDER.resolve()}')
print(f'DIRE classifier checkpoint override: {DIRE_CLASSIFIER_CHECKPOINT_PATH}')


In [ ]:
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}

def list_image_paths(folder: Path) -> list[Path]:
    if not folder.exists() or not folder.is_dir():
        raise ValueError(f'Image folder does not exist or is not a directory: {folder}')
    image_paths = sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS],
        key=lambda p: p.name.lower(),
    )
    if not image_paths:
        raise ValueError(f'No image files found in {folder}.')
    return image_paths

def load_images(paths: list[Path]) -> list[Image.Image]:
    images: list[Image.Image] = []
    for path in paths:
        with Image.open(path) as img:
            images.append(img.copy())
    return images

image_paths = list_image_paths(IMAGE_FOLDER)
images = load_images(image_paths)
image_names = [p.name for p in image_paths]

print(f'Loaded {len(images)} images')


In [ ]:
# Instantiate evaluators once and reuse
detectors: list[tuple[str, object]] = []
init_times: dict[str, float] = {}

def _add_detector(name: str, factory):
    start_time = time.perf_counter()
    detector = factory()
    elapsed = time.perf_counter() - start_time
    detectors.append((name, detector))
    init_times[name] = elapsed
    print(f'Initialized {name} in {elapsed:.2f}s')

if ENABLE_SDXL_DETECTOR:
    _add_detector(
        'SDXL Detector (Organika/sdxl-detector)',
        lambda: AIDetectionImageEvaluator(device=DEVICE, model='Organika/sdxl-detector'),
    )
if ENABLE_SSP:
    _add_detector(
        'SSP',
        lambda: SSPAIDetectionImageEvaluator(device=DEVICE),
    )
if ENABLE_DIRE_OFFICIAL:
    _add_detector(
        'DIRE Official (ADM/DDIM, paper)',
        lambda: DIREAIDetectionImageEvaluator(
            device=DIRE_DEVICE,
            backend='adm-ddim-official',
            classifier_checkpoint_path=DIRE_CLASSIFIER_CHECKPOINT_PATH,
            download_if_missing=DIRE_DOWNLOAD_IF_MISSING,
        ),
    )
if ENABLE_DIRE_SDXL_EXPERIMENTAL:
    _add_detector(
        'DIRE SDXL Adaptation (experimental)',
        lambda: DIREAIDetectionImageEvaluator(
            device=DIRE_DEVICE,
            backend='sdxl-turbo-experimental',
            classifier_checkpoint_path=DIRE_CLASSIFIER_CHECKPOINT_PATH,
            download_if_missing=DIRE_DOWNLOAD_IF_MISSING,
        ),
    )

if not detectors:
    raise ValueError('No detector enabled. Set at least one ENABLE_* flag to True.')

print('\nInitialization summary:')
for name, detector in detectors:
    print(f'- {name}: {init_times[name]:.2f}s')
    if isinstance(detector, DIREAIDetectionImageEvaluator):
        print(f'  requested_device={detector.requested_device} runtime_device={detector.device}')
        print(f'  classifier_checkpoint={detector._classifier_checkpoint_path}')
        if detector.backend == 'adm-ddim-official':
            print(f'  reconstruction_model={detector._reconstruction_model_path}')


In [ ]:
scores: dict[str, list[float]] = {name: [] for name, _ in detectors}
timings: dict[str, float] = {name: 0.0 for name, _ in detectors}
per_image_timings: dict[str, list[float]] = {name: [] for name, _ in detectors}
dire_diagnostics: dict[str, list[dict]] = {
    name: [] for name, evaluator in detectors if isinstance(evaluator, DIREAIDetectionImageEvaluator)
}

def _score_to_float(value: object) -> float:
    if isinstance(value, torch.Tensor):
        return float(value.detach().to(device='cpu', dtype=torch.float32).item())
    if isinstance(value, np.generic):
        return float(value.item())
    return float(value)

for image in images:
    batch = ImageSolutionData(images=[image])
    for name, evaluator in detectors:
        start_time = time.perf_counter()
        if isinstance(evaluator, DIREAIDetectionImageEvaluator):
            diagnostic = evaluator.evaluate_image_with_diagnostics(image)
            score = float(diagnostic['human_score'])
            dire_diagnostics[name].append(diagnostic)
        else:
            score = _score_to_float(evaluator.evaluate(batch))
        elapsed = time.perf_counter() - start_time
        timings[name] += elapsed
        per_image_timings[name].append(elapsed)
        scores[name].append(score)

for name, _ in detectors:
    mean_score = np.mean(scores[name])
    total_time = timings[name]
    avg_time = total_time / len(images)
    print(f"{name}: mean={mean_score:.2f} | total_time={total_time:.2f}s | avg_per_image={avg_time:.2f}s")
    if name in dire_diagnostics and dire_diagnostics[name]:
        mean_syn = np.mean([d['synthetic_probability'] for d in dire_diagnostics[name]])
        mean_logit = np.mean([d['logit'] for d in dire_diagnostics[name]])
        print(
            f"  DIRE diagnostics: mean_p_synth={mean_syn:.4f} mean_logit={mean_logit:.4f} "
            f"runtime_device={dire_diagnostics[name][-1]['runtime_device']} fallback={dire_diagnostics[name][-1]['used_cpu_fallback']}"
        )


In [ ]:
num_images = len(images)
fig, axs = plt.subplots(2, num_images, figsize=(max(14, num_images * 3.0), 6), gridspec_kw={'height_ratios': [3, 1]})
if num_images == 1:
    axs = np.array([[axs[0]], [axs[1]]])

detector_names = [name for name, _ in detectors]
for idx, (image, image_name) in enumerate(zip(images, image_names)):
    ax_img = axs[0, idx]
    ax_img.imshow(image)
    ax_img.axis('off')
    ax_img.set_title(image_name, fontsize=10)

    ax_text = axs[1, idx]
    ax_text.axis('off')
    lines = []
    for name in detector_names:
        line = f"{name}: H={scores[name][idx]:.2f}% t={per_image_timings[name][idx]:.2f}s"
        if name in dire_diagnostics:
            diag = dire_diagnostics[name][idx]
            line += (
                f" p_s={diag['synthetic_probability']:.3f} logit={diag['logit']:.2f} "
                f"d[min/mean/max]={diag['dire_min']:.3f}/{diag['dire_mean']:.3f}/{diag['dire_max']:.3f}"
            )
        lines.append(line)
    ax_text.text(0.5, 0.5, '\n'.join(lines), ha='center', va='center', fontsize=8, family='monospace')

fig.suptitle('Per-image AI-detection fitness comparison', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
detector_names = [name for name, _ in detectors]
num_detectors = len(detector_names)
x = np.arange(num_images)
width = 0.8 / max(1, num_detectors)

fig, ax = plt.subplots(figsize=(max(8, num_images * 1.3), 4.8))
for det_idx, name in enumerate(detector_names):
    offset = (det_idx - (num_detectors - 1) / 2.0) * width
    ax.bar(x + offset, scores[name], width=width, label=name)

ax.set_ylim(0, 100)
ax.set_ylabel('Human-likeliness fitness (%)')
ax.set_title('Detector score comparison per image')
ax.set_xticks(x)
ax.set_xticklabels(image_names, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()
